## Get observations for each state

In [ ]:
from pyinaturalist import *
import pandas as pd




ambystoma = 26721 # taxon id (Ambystoma genus)
states = { # place ids
    "Ohio": 31,
    "Michigan": 29,
    "Indiana": 20,
    "Pennsylvania": 42,
    "Kentucky": 26,
    "West Virginia": 33
}


observations = {"results": [], "total_results": 0}
for state in states:
    response = get_observations(
        taxon_id=ambystoma,
        place_id=states[state],
        quality_grade='research',
        captive=False,   # exclude captive/zoo observations
        page="all",      # fetch all pages
    )
    for obs in response['results']:
        obs['state'] = state  # tag each record with its state
    observations['results'].extend(response['results'])
    observations['total_results'] += response['total_results']
    print(f"{state}: {response['total_results']} research-grade Ambystoma observations")

print(f"Total: {observations['total_results']} research-grade Ambystoma observations across {len(states)} states")
pprint(observations['results'][:5])

In [ ]:
df = pd.DataFrame(observations['results'])
df.head()

# Sightings Visualization

### Empty six-state raster canvas

Builds the same OSMnx six-state region and locks the same 30 m-aligned `EPSG:5070` bounds as `cont_SHC_raster_prep`, `NLCD_raster_prep2`, `pH_raster_prep`, and `AWS_raster_prep`, writes an empty (all-zero) GeoTIFF on that grid, and plots the observations as points on it.

State outlines in the plot use the Census cartographic boundary, which follows the Great Lakes shoreline; OSM's state polygons are the legal state lines and run out into the lakes, which made Michigan look like a blob cut by straight lines through open water.


In [ ]:
import os
import urllib.request
import numpy as np
import geopandas as gpd
import osmnx as ox
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Same six states, target CRS, and 30 m grid as the raster prep notebooks
states = ["Ohio", "Michigan", "Indiana", "Pennsylvania", "Kentucky", "West Virginia"]
DST_CRS = "EPSG:5070"
GRID_RES = 30.0  # metres, matching the 30 m master grid
EMPTY_RASTER_PATH = "/kaggle/working/sightings_grid_5070.tif"

ox.settings.cache_folder = "/kaggle/working/cache"

region_queries = [f"{state}, USA" for state in states]
regions_gdf = ox.geocode_to_gdf(region_queries)

# Project to EPSG:5070 BEFORE unioning/buffering so geometry ops happen in meters
regions_5070 = regions_gdf.to_crs(DST_CRS)
master_boundary = regions_5070.union_all().buffer(0)

# Exact same 30 m-aligned bounding box logic as the other prep notebooks
minx, miny, maxx, maxy = master_boundary.bounds
te_bounds = (
    int(minx // GRID_RES) * GRID_RES,
    int(miny // GRID_RES) * GRID_RES,
    int((maxx // GRID_RES) + 1) * GRID_RES,
    int((maxy // GRID_RES) + 1) * GRID_RES,
)
width = int((te_bounds[2] - te_bounds[0]) / GRID_RES)
height = int((te_bounds[3] - te_bounds[1]) / GRID_RES)
transform = from_origin(te_bounds[0], te_bounds[3], GRID_RES, GRID_RES)

print(f"Grid: {width} x {height} px @ {GRID_RES:g} m | CRS {DST_CRS}")
print(f"Bounds: {te_bounds}")

In [ ]:
# Empty (all-zero) canvas on that grid. Blocks are written one at a time and
# deflate-compressed, so a grid of 2e9 zero pixels becomes a small file.
profile = {
    "driver": "GTiff",
    "dtype": "uint8",
    "nodata": 0,
    "width": width,
    "height": height,
    "count": 1,
    "crs": DST_CRS,
    "transform": transform,
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
    "compress": "deflate",
    "BIGTIFF": "IF_SAFER",
}

with rasterio.open(EMPTY_RASTER_PATH, "w", **profile) as dst:
    for _, window in dst.block_windows(1):
        dst.write(np.zeros((window.height, window.width), dtype="uint8"), 1, window=window)

print(f"Empty raster written: {EMPTY_RASTER_PATH}")
print(f"  {width} x {height} px | CRS {DST_CRS} | nodata 0 | "
      f"{os.path.getsize(EMPTY_RASTER_PATH) / 1e6:.1f} MB on disk")

In [ ]:
def extract_lonlat(record):
    """Pull (lon, lat) from a pyinaturalist record (geojson first, then location string)."""
    geo = record.get("geojson")
    if isinstance(geo, dict) and geo.get("coordinates"):
        lon, lat = geo["coordinates"][:2]
        return lon, lat
    loc = record.get("location")
    if isinstance(loc, str) and "," in loc:
        lat, lon = (float(v) for v in loc.split(","))
        return lon, lat
    return np.nan, np.nan


coords = [extract_lonlat(obs) for obs in observations["results"]]
lons = np.array([c[0] for c in coords], dtype="float64")
lats = np.array([c[1] for c in coords], dtype="float64")
valid = np.isfinite(lons) & np.isfinite(lats)

sightings = gpd.GeoDataFrame(
    df.iloc[np.flatnonzero(valid)].reset_index(drop=True),
    geometry=gpd.points_from_xy(lons[valid], lats[valid]),
    crs="EPSG:4326",
).to_crs(DST_CRS)

# Keep only points inside the six-state boundary (a few records land in
# neighbouring states or outside it after obscured coordinates are randomised).
sightings = sightings[sightings.within(master_boundary)].copy()
print(f"{int(valid.sum())} sightings with coordinates -> {len(sightings)} inside the region "
      f"({int(valid.sum()) - len(sightings)} dropped)")

In [ ]:
# OSM's state polygons are the legal state lines, and for Michigan (and Ohio /
# Pennsylvania) those run out into the Great Lakes to the international border.
# Drawn as-is, Michigan is a blob cut by straight lines through Lakes Michigan,
# Huron, Superior and Erie. The Census cartographic boundary follows the
# shoreline instead, so the states look clipped to the coast. This only changes
# what is drawn: the grid, bounds and cutline above are untouched.
CENSUS_STATES_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"
)
CENSUS_ZIP = "/kaggle/working/cb_2023_us_state_500k.zip"
if not os.path.exists(CENSUS_ZIP):
    urllib.request.urlretrieve(CENSUS_STATES_URL, CENSUS_ZIP)

state_codes = ["OH", "MI", "IN", "PA", "KY", "WV"]
state_outlines = gpd.read_file(f"zip://{CENSUS_ZIP}")
state_outlines = (
    state_outlines[state_outlines["STUSPS"].isin(state_codes)]
    .to_crs(DST_CRS)
    .set_index("STUSPS")
    .loc[state_codes]
)

# Decimated read of the empty canvas for the background image: drawing all
# 2e9 pixels at full resolution would gain nothing at plot size.
with rasterio.open(EMPTY_RASTER_PATH) as src:
    preview = src.read(
        1,
        out_shape=(max(1, src.height // 100), max(1, src.width // 100)),
        resampling=Resampling.nearest,
    )
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(preview, extent=extent, cmap="Greys", origin="upper")
state_outlines.boundary.plot(ax=ax, color="black", linewidth=0.8)

colors = plt.cm.tab10(np.linspace(0, 1, len(states)))
handles = []
for color, state in zip(colors, states):
    pts = sightings[sightings["state"] == state]
    ax.scatter(pts.geometry.x, pts.geometry.y, s=4, color=color, alpha=0.6, linewidths=0)
    handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
                          markersize=6, label=f"{state} ({len(pts)})"))

ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_aspect("equal")
ax.set_xlabel("Easting (m, EPSG:5070)")
ax.set_ylabel("Northing (m, EPSG:5070)")
ax.set_title("Ambystoma sightings on the six-state 30 m master grid")
ax.legend(handles=handles, loc="lower right")
plt.tight_layout()
plt.show()


### Prep the presence points for MaxEnt

Plan steps 1 and 2 of `MaxEnt_plan.md`, in three passes: drop records that do not
mean what they appear to mean, drop points that sit on a hole in the covariate
stack, then collapse duplicates and thin.

The order matters. Cleaning first means the thinning is not spent separating
records that were never usable, and thinning last means every retained point is
already known to be scoreable, so the final count is the real sample size.

Everything downstream (background points, spatial chunks, cross validation) is
built from the `presences` table this section produces.


In [ ]:
# ---- Plan step 2a: drop records that cannot be trusted to 30 m.
#
# iNaturalist relocates obscured and private observations (often by ~0.2 degrees,
# tens of kilometres), so those coordinates are not where the animal was. The
# `obscured` field is the public-coordinate flag, and `geoprivacy` /
# `taxon_geoprivacy` say why it was hidden: the threatened members of this genus
# are exactly the ones that get hidden, so this filter is not evenly spread
# across species.
#
# A stated accuracy radius coarser than the covariate cell means the record could
# be in a different habitat patch altogether.
#
# A missing accuracy radius is kept. It is the normal case in this API response
# (the field is only filled when the observer or the device supplied it), so
# dropping it would discard most of the dataset for a reason that says more about
# the upload than about the record.
MAX_POSITIONAL_ACCURACY_M = 250.0


def column_or_default(name, default):
    """The column if the API returned it, otherwise a column of `default`."""
    if name in sightings:
        return sightings[name]
    return pd.Series(default, index=sightings.index)


privacy_hidden = (
    column_or_default("obscured", False).fillna(False).astype(bool)
    | column_or_default("geoprivacy", None).isin(["obscured", "private"])
    | column_or_default("taxon_geoprivacy", None).isin(["obscured", "private"])
)
accuracy_m = pd.to_numeric(
    column_or_default("positional_accuracy", np.nan), errors="coerce"
)
too_coarse = accuracy_m > MAX_POSITIONAL_ACCURACY_M

sightings_clean = sightings[~privacy_hidden & ~too_coarse].copy()

print(f"record quality : {len(sightings)} -> {len(sightings_clean)}  "
      f"(dropped {int(privacy_hidden.sum())} relocated for privacy, "
      f"{int((too_coarse & ~privacy_hidden).sum())} coarser than "
      f"{MAX_POSITIONAL_ACCURACY_M:g} m)")
print(f"                 {int(accuracy_m[sightings_clean.index].isna().sum())} kept "
      f"records state no accuracy radius")


In [ ]:
# ---- Plan step 1b: keep only points where every covariate has a value.
#
# Being inside the state border is not enough. The covariates were cut to the
# six-state line, and the water-sensitive layers mark lakes as nodata, so a
# record can pass the border test and still land on a pixel the model cannot
# score. Left in, those points would either vanish when the design matrix is
# built or be treated as background, both of which corrupt the result quietly.
#
# Only a few thousand points need checking, so each raster is sampled at the
# points instead of intersecting a stack of 2-billion-pixel masks.
COVARIATE_PATHS = [
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/land_cover_clipped_reproj.tiff",                   # land cover classes
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/FctImp_clipped_reproj.tiff",                       # NLCD forest / impervious
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/canopy_cover_clipped_reproj.tiff",                 # tree canopy percent
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/gnatsgo_aws0_150_clipped_reproj.tiff",             # available water storage
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/polaris_ph_mean_0_5_clipped_reproj.tiff",          # soil pH, 0-5 cm
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/polaris_ksat_mean_0_5_log10_clipped_reproj.tiff",  # log10 Ksat, 0-5 cm
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/six_state_twi_aligned.tif",                        # wetness index
    "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/wetland_distance.tiff",                            # distance to wetland
]


def covariate_values_exist(src, xs, ys):
    """True where a point is inside the raster and its pixel is not nodata."""
    ok = np.zeros(len(xs), dtype=bool)
    inside = (
        (xs >= src.bounds.left) & (xs < src.bounds.right)
        & (ys > src.bounds.bottom) & (ys <= src.bounds.top)
    )
    idx = np.flatnonzero(inside)
    if idx.size:
        sampled = np.array(
            [v[0] for v in src.sample(zip(xs[idx], ys[idx]))], dtype="float64"
        )
        ok[idx] = np.isfinite(sampled)
        if src.nodata is not None:
            ok[idx] &= sampled != src.nodata
    return ok


geom_x = sightings_clean.geometry.x.to_numpy()
geom_y = sightings_clean.geometry.y.to_numpy()
on_ground = np.ones(len(sightings_clean), dtype=bool)
layers_checked = 0
for path in COVARIATE_PATHS:
    if not os.path.exists(path):
        print(f"  missing covariate, not checked: {path}")
        continue
    with rasterio.open(path) as src:
        on_ground &= covariate_values_exist(src, geom_x, geom_y)
    layers_checked += 1

if layers_checked == 0:
    print("WARNING: no covariate rasters found, so the nodata test was skipped")

presences = sightings_clean[on_ground].copy()
print(f"covariate data : {len(sightings_clean)} -> {len(presences)}  "
      f"(dropped {int((~on_ground).sum())} on nodata, {layers_checked} layers checked)")


In [ ]:
# ---- Plan step 2b: collapse duplicates, then enforce a minimum spacing.
#
# A pond visited every spring produces a dozen records a few metres apart. They
# are not a dozen independent facts about the habitat, they just outvote
# everywhere else, so exact duplicates and near-duplicates both have to go.
#
# Thinning at the 30 m covariate cell size would achieve nothing: adjacent pixels
# carry nearly identical land cover, soil and terrain. The spacing has to be on
# the order of how far these animals move and how fast the covariates change,
# which is kilometres. The plan's useful range is 1-5 km.
THIN_DISTANCE_M = 2000.0

# Oldest record wins, so the retained points do not depend on API ordering.
presences = presences.sort_values("id").reset_index(drop=True)

# Exact duplicates: one location entered twice, e.g. one sighting logged under two
# projects. Coordinates are rounded to the metre first so float noise cannot hide
# a duplicate.
presences["_x"] = np.round(presences.geometry.x, 0)
presences["_y"] = np.round(presences.geometry.y, 0)
presences = presences.drop_duplicates(subset=["_x", "_y"]).reset_index(drop=True)
n_deduped = len(presences)
px = presences["_x"].to_numpy()
py = presences["_y"].to_numpy()


def thin_min_distance(px, py, min_distance):
    """Keep a point only if no already-kept point is closer than min_distance.

    Points are bucketed into square cells one min_distance wide. Anything closer
    than min_distance must fall in the point's own cell or in one of the eight
    around it, so each point is compared against a handful of neighbours
    instead of all of them. Greedy, so the order of the input decides which of
    two close points survives.
    """
    r2 = min_distance * min_distance
    kept_cells = {}
    keep = np.zeros(len(px), dtype=bool)

    def too_close(x, y, col, row):
        for dc in (-1, 0, 1):
            for dr in (-1, 0, 1):
                for j in kept_cells.get((col + dc, row + dr), ()):
                    if (x - px[j]) ** 2 + (y - py[j]) ** 2 < r2:
                        return True
        return False

    for i in range(len(px)):
        col, row = int(px[i] // min_distance), int(py[i] // min_distance)
        if not too_close(px[i], py[i], col, row):
            keep[i] = True
            kept_cells.setdefault((col, row), []).append(i)
    return keep


keep = thin_min_distance(px, py, THIN_DISTANCE_M)
presences = presences[keep].drop(columns=["_x", "_y"]).reset_index(drop=True)

print(f"thinning       : {n_deduped} -> {len(presences)} points "
      f"({n_deduped - len(presences)} too close to a retained point)")

per_state = presences["state"].value_counts().reindex(states, fill_value=0)
print(f"presences      : {len(presences)} at >= {THIN_DISTANCE_M / 1000:g} km spacing")
print("                 " + "  ".join(f"{s} {int(n)}" for s, n in per_state.items()))


In [ ]:
# ---- Visualization: what the filters kept, and where.
#
# Left panel is every point that passed the record-quality and nodata tests, so
# it still contains the near-duplicates. Right panel is the final presence set,
# coloured by state. Reading them side by side shows how much of the apparent
# density was repeat visits to the same ponds rather than independent evidence
# of habitat, which is the thing the thinning step is spending points on.
print(f"{len(sightings)} raw sightings -> {len(sightings_clean)} usable "
      f"-> {len(presences)} modelled presences")

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
xlim = (te_bounds[0], te_bounds[2])
ylim = (te_bounds[1], te_bounds[3])

# Left: passed the filters, before thinning.
ax = axes[0]
state_outlines.boundary.plot(ax=ax, color="black", linewidth=0.6)
ax.scatter(
    sightings_clean.geometry.x, sightings_clean.geometry.y,
    s=3, color="0.45", alpha=0.5, linewidths=0,
)
ax.set_title(f"After quality and nodata filters ({len(sightings_clean)})")
ax.legend(
    handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor="0.45",
                    markersize=5, label="usable point")],
    loc="lower right",
)

# Right: the retained set, coloured by state with per-state counts.
ax = axes[1]
state_outlines.boundary.plot(ax=ax, color="black", linewidth=0.6)
colors = plt.cm.tab10(np.linspace(0, 1, len(states)))
handles = []
for color, state in zip(colors, states):
    pts = presences[presences["state"] == state]
    ax.scatter(pts.geometry.x, pts.geometry.y, s=6, color=color, alpha=0.8,
               linewidths=0)
    handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
                          markersize=6, label=f"{state} ({len(pts)})"))
ax.set_title(f"Presences after thinning to >= {THIN_DISTANCE_M / 1000:g} km "
             f"({len(presences)})")
ax.legend(handles=handles, loc="lower right")

for ax in axes:
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_xlabel("Easting (m, EPSG:5070)")
axes[0].set_ylabel("Northing (m, EPSG:5070)")

fig.suptitle("MaxEnt presence points: record filters and thinning", fontsize=14)
plt.tight_layout()
plt.show()


### Background points for MaxEnt

Plan step 3 of `MaxEnt_plan.md`. The background sample is the model's picture of an
ordinary landscape: not absences, just the yardstick the presences are compared
against. Two things about it are decided here and carried all the way through the
cross validation: it only touches pixels the covariates can score, and it is spread
at a constant density per unit area so that predictions from different folds are on
the same scale and can be stitched together without artefact seams.

---



In [ ]:
# ---- Plan step 3: background points.
#
# Background is the model's picture of an ordinary landscape. These points are not
# absences -- nothing here is known to be empty -- they are the yardstick the
# presences are compared against, and they also fix the scale that turns the raw
# model output into a score that means the same thing everywhere in the region.
#
# Two rules from the plan, both satisfied by the same trick:
#
# * Only where the covariates exist, so no draw is wasted and nothing has to be
#   dropped quietly at fit time.
# * Constant density per unit area over the whole region -- not proportional to
#   land area and not proportional to how large a chunk happens to be. If one part
#   of the region got twice the background density of another, every fold's scores
#   would be stretched differently and the stitched map would show seams that are
#   sampling artefacts rather than habitat.
#
# Drawing candidates uniformly over the full grid rectangle and keeping only those
# inside the region where every layer has a value gives exactly that: uniform over
# the rectangle, restricted to a subset, is still uniform over the subset.
#
# The chunk each point belongs to is attached in step 4; step 5 then removes the
# held-out chunk's background from that fold's training set, so the model never
# sees the landscape it is about to be scored on. That is a leak of the same kind
# as training on a presence inside the held-out chunk.
import shapely

N_BACKGROUND = 50_000
BACKGROUND_SEED = 0
MAX_CANDIDATES = 10 * N_BACKGROUND  # give up rather than loop forever on a broken mask

rng = np.random.default_rng(BACKGROUND_SEED)


def random_pixel_centres(rng, n):
    """n uniformly random pixel centres on the 30 m grid.

    Sampling pixels uniformly is sampling area uniformly: every pixel covers the
    same ground, so this is constant density by construction.
    """
    rows = rng.integers(0, height, n)
    cols = rng.integers(0, width, n)
    xs = transform.c + (cols + 0.5) * transform.a
    ys = transform.f + (rows + 0.5) * transform.e
    return xs, ys


bx, by = [], []
n_candidates = n_inside = n_kept = n_on_nodata = 0
layers_checked = 0
while n_kept < N_BACKGROUND and n_candidates < MAX_CANDIDATES:
    batch = min(N_BACKGROUND, MAX_CANDIDATES - n_candidates)
    xs, ys = random_pixel_centres(rng, batch)
    n_candidates += batch

    # Inside the six-state outline. Cheap, and it discards the large empty corners
    # of the bounding rectangle that the states never occupy (most of Lake
    # Michigan, Huron, Erie and Ontario, and everything beyond them).
    on_land = shapely.contains_xy(master_boundary, xs, ys)
    n_inside += int(on_land.sum())
    xs, ys = xs[on_land], ys[on_land]

    # Present in every covariate layer: lakes and any gap in the stack are nodata.
    usable = np.ones(len(xs), dtype=bool)
    layers_checked = 0
    for path in COVARIATE_PATHS:
        if not os.path.exists(path):
            continue
        with rasterio.open(path) as src:
            usable &= covariate_values_exist(src, xs, ys)
        layers_checked += 1
    n_on_nodata += int((~usable).sum())

    bx.append(xs[usable])
    by.append(ys[usable])
    n_kept = sum(len(v) for v in bx)
    print(f"  draw {n_candidates:>7,}: {n_inside:>7,} inside the region, "
          f"{n_kept:>7,} with covariate data")

if layers_checked == 0:
    print("WARNING: no covariate rasters found, so only the state outline was checked")
    print("         -- some background points probably sit on water or nodata")
if n_kept < N_BACKGROUND:
    print(f"WARNING: only {n_kept:,} of {N_BACKGROUND:,} background points found "
          f"after {n_candidates:,} draws")

# A final batch can overshoot; keep the first N in draw order, which is still a
# uniform sample.
bx = np.concatenate(bx)[:N_BACKGROUND]
by = np.concatenate(by)[:N_BACKGROUND]

background = gpd.GeoDataFrame(
    {"id": np.arange(len(bx))},
    geometry=gpd.points_from_xy(bx, by),
    crs=DST_CRS,
)

region_km2 = master_boundary.area / 1e6
print(f"background     : {len(background):,} points")
print(f"                 {n_candidates:,} drawn, "
      f"{n_candidates - n_inside:,} outside the region, "
      f"{n_on_nodata:,} on missing covariates "
      f"({layers_checked}/{len(COVARIATE_PATHS)} layers checked)")
if len(background):
    print(f"density        : ~1 point per {region_km2 / len(background):.1f} km2 "
          f"of the {region_km2:,.0f} km2 region")


In [ ]:
# ---- Presences and background on the same map.
#
# Background is drawn underneath in a cool, light colour and the presences on top
# in a warm, dark one, because the question here is not "which state is which" but
# whether the two samples describe the same region: the background should cover
# the whole study area evenly, with no hole where a covariate layer is missing and
# no pile-up along the state lines, while the presences pick out where the animals
# actually were recorded.
#
# This cannot show whether the background density is constant -- at 50k points
# over the six states the dots merge into a wash, which is the point. It shows
# coverage, not spacing. The spacing is guaranteed by how the points were drawn.
fig, ax = plt.subplots(figsize=(12, 10))

state_outlines.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.scatter(
    background.geometry.x, background.geometry.y,
    s=1.2, color="tab:blue", alpha=0.25, linewidths=0,
)
ax.scatter(
    presences.geometry.x, presences.geometry.y,
    s=6, color="crimson", alpha=0.85, linewidths=0,
)

ax.set_xlim(te_bounds[0], te_bounds[2])
ax.set_ylim(te_bounds[1], te_bounds[3])
ax.set_aspect("equal")
ax.set_xlabel("Easting (m, EPSG:5070)")
ax.set_ylabel("Northing (m, EPSG:5070)")
ax.set_title(f"{len(presences):,} presences over {len(background):,} background points")

handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson",
           markersize=6, label=f"presences ({len(presences):,})"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue",
           markersize=6, label=f"background ({len(background):,})"),
]
ax.legend(handles=handles, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()


### Spatial chunks for leave-one-chunk-out evaluation

Plan step 4 of `MaxEnt_plan.md`. Two decisions are made here: the shape of the
chunks, and where the prediction compute goes.

**Shape.** A regular grid of rectangles, not clusters of survey points. A raster can
only be cut along a rectangle, so a cluster-shaped chunk could never be predicted on;
and equal blocks give every fold a predictable share of the region and of the compute.
The plan aims at 6-12 chunks, which is a trade-off: fewer chunks means each model is
missing a large slab of the region and stops resembling the final full-region fit,
more chunks means each fold has too few presences to fit anything. The candidate sweep
below counts presences per chunk for several grids, because that is the number that
decides it — and a chunk with no presences cannot be a fold at all.

**Compute.** Predicting the full region is roughly two billion 30 m pixels, and doing
that once per fold is the dominant cost of the whole project. So the rasters are read
and predicted a chunk at a time, in row strips sized to stay in memory, and only for
the chunk being predicted. The window helpers below are what step 5 will read through.

The grid is snapped to the 30 m pixel lattice — every split line falls on a pixel
edge — so a chunk is a whole number of pixels and cutting a covariate at a chunk
boundary is a window read with no resampling and no half-pixel shift.

From here on every point carries the id of its chunk. Step 5 removes a fold's held-out
chunk (its presences *and* its background) from training, predicts into that chunk,
and drops a 5-10 km band inside its edge from the scoring but not from the map.


In [ ]:
# ---- Plan step 4a: pixel-aligned chunk grid, and the audit that chooses it.
#
# Two requirements from the plan meet in this cell. Chunks have to be rectangles,
# because a raster can only be cut along a rectangle, and they have to be regular,
# because then each fold gets a predictable share of the region and of the compute.
#
# The split lines are snapped to the 30 m pixel lattice rather than to round
# coordinates. A chunk boundary is then a pixel edge, so cutting a covariate there
# is a pure window read: no resampling, no half-pixel shift, and an exact pixel
# width and height to size the prediction strips with.
#
# Chunk ids are row-major (north to south, then west to east). They are fixed here
# and saved alongside the points because the fold loop and the per-fold prediction
# rasters key off them.
CHUNK_NODATA_ID = -1
MIN_PRESENCES_PER_CHUNK = 25

presence_x = presences.geometry.x.to_numpy()
presence_y = presences.geometry.y.to_numpy()
background_x = background.geometry.x.to_numpy()
background_y = background.geometry.y.to_numpy()


def pixel_edges(n_cells, n_px):
    """`n_cells` spans of `n_px` pixels, as equal as whole pixels allow."""
    return np.round(np.linspace(0, n_px, n_cells + 1)).astype(int)


def rectangles_from_cells(cell_sets, cols, rows):
    """Chunk rectangles for sets of grid cells, each set a filled rectangle."""
    xe = pixel_edges(cols, width)
    ye = pixel_edges(rows, height)  # row offsets run south from the top edge
    rects = []
    for cellset in cell_sets:
        rs = [r for r, _ in cellset]
        cs = [c for _, c in cellset]
        r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)
        rects.append({
            "col_off": int(xe[c0]),
            "row_off": int(ye[r0]),
            "width_px": int(xe[c1 + 1] - xe[c0]),
            "height_px": int(ye[r1 + 1] - ye[r0]),
            "left": te_bounds[0] + xe[c0] * GRID_RES,
            "right": te_bounds[0] + xe[c1 + 1] * GRID_RES,
            "top": te_bounds[3] - ye[r0] * GRID_RES,
            "bottom": te_bounds[3] - ye[r1 + 1] * GRID_RES,
            "_first": r0 * cols + c0,
        })
    rects = (
        pd.DataFrame(rects)
        .sort_values("_first")
        .drop(columns="_first")
        .reset_index(drop=True)
    )
    rects["chunk_id"] = np.arange(len(rects))
    rects["area_km2"] = rects["width_px"] * rects["height_px"] * GRID_RES**2 / 1e6
    return rects


def grid_rectangles(cols, rows):
    """A `cols` x `rows` grid of rectangles that tiles the whole grid box."""
    cells = [{(r, c)} for r in range(rows) for c in range(cols)]
    return rectangles_from_cells(cells, cols, rows)


def assign_chunk_ids(xs, ys, rects):
    """Chunk id per point, or CHUNK_NODATA_ID for a point that is off the grid.

    Rectangles include their left and bottom edges and exclude their right and
    top, so a point exactly on a shared edge falls in one chunk and only one.
    """
    ids = np.full(len(xs), CHUNK_NODATA_ID, dtype=int)
    for chunk in rects.itertuples():
        inside = (
            (xs >= chunk.left) & (xs < chunk.right)
            & (ys > chunk.bottom) & (ys <= chunk.top)
        )
        ids[inside] = chunk.chunk_id
    return ids


CANDIDATE_GRIDS = [(2, 3), (3, 2), (3, 3), (4, 3), (3, 4), (4, 4)]


def audit_grid(cols, rows):
    """Presences and background per chunk for one candidate grid, before merging.

    Nothing is merged at this stage: the point is to see the raw grid, including
    the chunks that come out empty. The background count is shown next to the
    presence count because a chunk can be full of landscape and still hold almost
    no records -- that is uneven survey effort, not unsuitable habitat, and the
    fold it produces would say more about the sampling than about the species.
    """
    rects = grid_rectangles(cols, rows)
    p_ids = assign_chunk_ids(presence_x, presence_y, rects)
    b_ids = assign_chunk_ids(background_x, background_y, rects)
    n_p = np.bincount(p_ids[p_ids >= 0], minlength=len(rects))
    n_b = np.bincount(b_ids[b_ids >= 0], minlength=len(rects))
    return {
        "cols": cols,
        "rows": rows,
        "chunks": len(rects),
        "km2": int(round(rects["area_km2"].median())),
        "pres_min": int(n_p.min()),
        "pres_mean": int(round(n_p.mean())),
        "pres_max": int(n_p.max()),
        "pres_empty": int((n_p == 0).sum()),
        "pres_small": int((n_p < MIN_PRESENCES_PER_CHUNK).sum()),
        "bg_min": int(n_b.min()),
        "off_grid": int((p_ids < 0).sum() + (b_ids < 0).sum()),
    }


candidate_audit = pd.DataFrame([audit_grid(c, r) for c, r in CANDIDATE_GRIDS])
print("Candidate chunk grids (nothing merged yet):")
print(candidate_audit.to_string(index=False))
print()
print("Read pres_min / pres_empty / pres_small first: an empty chunk cannot be a")
print("fold, and a chunk with a handful of presences gives a fold whose metrics")
print("are noise. Fewer chunks is the cheap fix. off_grid must be 0.")


In [ ]:
# ---- Plan step 4b: choose one grid, fold dead chunks into their neighbours,
# attach every point to a chunk, and save the artefacts step 5 needs.
#
# 3 x 2: on the real presences the audit above left 3 x 2 as the only candidate
# with no empty and no under-populated chunk (156-778 presences per chunk), so
# nothing has to be merged and every fold holds an equal sixth of the region.
# 3 x 3 came out one cell short in the dead north-east corner, and absorbing that
# cell doubled the chunk below it to 22% of the region; 4 x 3 and 3 x 4 both
# started with two empty cells. Change these two numbers only -- everything
# below is derived from them.
CHUNK_COLS, CHUNK_ROWS = 3, 2


def merge_small_chunks(cols, rows, presence_ids, min_presences=MIN_PRESENCES_PER_CHUNK):
    """Fold under-populated chunks into an adjacent chunk, keeping rectangles.

    A chunk with no presences cannot be a fold: there is nothing to score, and
    holding it out only deletes training area. A chunk with a handful of
    presences gives a fold whose numbers are noise. The fix the plan calls for is
    fewer chunks, and the cheapest version is to grow a neighbour's edge across
    the dead cell rather than rebuild the whole grid.

    A merge is only accepted when the union of the two chunks is still a filled
    rectangle; otherwise the raster could no longer be cut along it. Returns the
    merged cell sets, which `rectangles_from_cells` turns back into rectangles,
    and a log of what moved. `presence_ids` are ids on the *unmerged* grid, since
    merging changes the numbering.
    """
    owner = np.arange(rows * cols).reshape(rows, cols)
    cells = {i: {(i // cols, i % cols)} for i in range(rows * cols)}
    counts = {i: int((presence_ids == i).sum()) for i in range(rows * cols)}
    merges = []

    def is_rectangle(cellset):
        rs = [r for r, _ in cellset]
        cs = [c for _, c in cellset]
        return (max(rs) - min(rs) + 1) * (max(cs) - min(cs) + 1) == len(cellset)

    while len(cells) > 1:
        # Worst first, so the emptiest chunk is dissolved before anything else.
        small = sorted(
            (i for i, n in counts.items() if n < min_presences),
            key=lambda i: counts[i],
        )
        if not small:
            break

        merged_any = False
        for victim in small:
            neighbours = set()
            for r, c in cells[victim]:
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < rows and 0 <= cc < cols and owner[rr, cc] != victim:
                        neighbours.add(int(owner[rr, cc]))
            options = [o for o in neighbours if is_rectangle(cells[victim] | cells[o])]
            if not options:
                continue  # an L-shape would break the raster cut

            # Prefer the biggest neighbour, then the best populated, so the dead
            # cell disappears into a chunk that already carries the fold.
            target = max(options, key=lambda o: (len(cells[o]), counts[o]))
            n_victim = counts[victim]
            for r, c in cells[victim]:
                owner[r, c] = target
            cells[target] |= cells[victim]
            counts[target] += n_victim
            merges.append({
                "from_chunk": victim,
                "into_chunk": target,
                "presences_moved": n_victim,
            })
            del cells[victim], counts[victim]
            merged_any = True
            break

        if not merged_any:
            break

    return list(cells.values()), pd.DataFrame(merges)


raw_chunks = grid_rectangles(CHUNK_COLS, CHUNK_ROWS)
raw_presence_ids = assign_chunk_ids(presence_x, presence_y, raw_chunks)
merged_cells, merge_log = merge_small_chunks(CHUNK_COLS, CHUNK_ROWS, raw_presence_ids)
chunks = rectangles_from_cells(merged_cells, CHUNK_COLS, CHUNK_ROWS)

# Attach the chunk to every point. Background gets its chunk for the same reason
# presences do: step 5 removes the held-out chunk's background from training.
presences["chunk_id"] = assign_chunk_ids(presence_x, presence_y, chunks)
background["chunk_id"] = assign_chunk_ids(background_x, background_y, chunks)

if (presences["chunk_id"] < 0).any() or (background["chunk_id"] < 0).any():
    print(f"WARNING: {int((presences['chunk_id'] < 0).sum())} presences and "
          f"{int((background['chunk_id'] < 0).sum())} background points fell outside "
          f"the chunk grid; they would be dropped from the folds")

chunk_table = chunks.copy()
chunk_table["n_presences"] = np.bincount(
    presences["chunk_id"].to_numpy().clip(min=0), minlength=len(chunks)
)
chunk_table["n_background"] = np.bincount(
    background["chunk_id"].to_numpy().clip(min=0), minlength=len(chunks)
)
print(f"chosen grid    : {CHUNK_COLS} x {CHUNK_ROWS} -> {len(chunks)} chunks "
      f"({len(raw_chunks)} before merging, {len(merge_log)} absorbed)")
if len(merge_log):
    print("merges         : " + ", ".join(
        f"grid cell {m.from_chunk} -> {m.into_chunk} "
        f"({m.presences_moved} presences)"
        for m in merge_log.itertuples()
    ) + f" -- cell numbers on the {CHUNK_COLS} x {CHUNK_ROWS} grid, before "
          "the surviving chunks are renumbered 0..n-1")
print(f"chunk size     : {chunk_table['area_km2'].min():,.0f}-"
      f"{chunk_table['area_km2'].max():,.0f} km2 "
      f"({chunk_table['width_px'].min():,}-{chunk_table['width_px'].max():,} px wide, "
      f"{chunk_table['height_px'].min():,}-{chunk_table['height_px'].max():,} px tall)")
print(f"presences      : {chunk_table['n_presences'].min()}-"
      f"{chunk_table['n_presences'].max()} per chunk "
      f"(mean {chunk_table['n_presences'].mean():.0f}, "
      f"{len(presences):,} total at >= {THIN_DISTANCE_M / 1000:g} km spacing)")
print(f"background     : {chunk_table['n_background'].min():,}-"
      f"{chunk_table['n_background'].max():,} per chunk "
      f"({len(background):,} total)")
if (chunk_table["n_presences"] < MIN_PRESENCES_PER_CHUNK).any():
    print(f"WARNING: {int((chunk_table['n_presences'] < MIN_PRESENCES_PER_CHUNK).sum())} "
          f"chunk(s) still hold fewer than {MIN_PRESENCES_PER_CHUNK} presences and "
          f"cannot be absorbed into a rectangle; use fewer chunks")
print()
print(chunk_table[[
    "chunk_id", "left", "bottom", "width_px", "height_px",
    "area_km2", "n_presences", "n_background",
]].round(0).to_string(index=False))

# ---- Artefacts for step 5. The per-fold loop needs to know which points and
# which raster window belong to which chunk, so both are written out rather than
# rebuilt from scratch: a fold must be reproducible from the files alone.
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CHUNK_GRID_PATH = os.path.join(OUT_DIR, "chunks.gpkg")
CHUNK_TABLE_PATH = os.path.join(OUT_DIR, "chunks.csv")
PRESENCES_CHUNKED_PATH = os.path.join(OUT_DIR, "presences_chunked.gpkg")
BACKGROUND_CHUNKED_PATH = os.path.join(OUT_DIR, "background_chunked.gpkg")
CHUNK_SETTINGS_PATH = os.path.join(OUT_DIR, "chunk_settings.csv")

chunks_gdf = gpd.GeoDataFrame(
    chunk_table,
    geometry=[
        shapely.box(c.left, c.bottom, c.right, c.top) for c in chunk_table.itertuples()
    ],
    crs=DST_CRS,
)
for path in (CHUNK_GRID_PATH, PRESENCES_CHUNKED_PATH, BACKGROUND_CHUNKED_PATH):
    if os.path.exists(path):
        os.remove(path)  # a GPKG layer cannot simply be appended to

chunks_gdf.to_file(CHUNK_GRID_PATH, driver="GPKG")
presences[["id", "state", "chunk_id", "geometry"]].to_file(
    PRESENCES_CHUNKED_PATH, driver="GPKG"
)
background[["id", "chunk_id", "geometry"]].to_file(
    BACKGROUND_CHUNKED_PATH, driver="GPKG"
)
chunk_table.to_csv(CHUNK_TABLE_PATH, index=False)

# Settings table: a map without its provenance cannot be reproduced or defended.
pd.DataFrame({
    "setting": [
        "crs", "grid_res_m", "grid_bounds", "grid_px",
        "chunk_cols", "chunk_rows", "n_chunks", "min_presences_per_chunk",
        "thin_distance_m", "n_presences", "n_background", "background_seed",
    ],
    "value": [
        DST_CRS, GRID_RES, str(tuple(te_bounds)), f"{width} x {height}",
        CHUNK_COLS, CHUNK_ROWS, len(chunks), MIN_PRESENCES_PER_CHUNK,
        THIN_DISTANCE_M, len(presences), len(background), BACKGROUND_SEED,
    ],
}).to_csv(CHUNK_SETTINGS_PATH, index=False)

for path in (CHUNK_GRID_PATH, CHUNK_TABLE_PATH, PRESENCES_CHUNKED_PATH,
             BACKGROUND_CHUNKED_PATH, CHUNK_SETTINGS_PATH):
    print(f"saved          : {path} ({os.path.getsize(path) / 1e6:.1f} MB)")


In [ ]:
# ---- Plan step 4d: export the chunks for the MaxEnt notebook.
#
# data_prep ends here. The model, the folds and the per-fold prediction rasters are
# built in a second notebook, and it must not rebuild any of this: if the two
# notebooks each derived their own grid, a rerun of one would silently move the
# folds of the other. So everything that crosses the boundary goes to disk -- the
# chunk table and the two point tables in the cell above, and here the covariate
# order, the raster-window helpers, and a manifest that names all of it.
#
# `chunk_io.py` is written first, then run in this notebook from the same text, so
# there is one copy of the window arithmetic: the copy the model notebook imports
# is the copy this notebook sized its strips with.
import json
from datetime import datetime, timezone

STRIP_TARGET_BYTES = 512 * 2 ** 20

CHUNK_IO_SOURCE = r'''"""Raster windows for one chunk, shared with the MaxEnt notebook.

Written by data_prep.ipynb (step 4d) and imported by the modelling notebook, so
the two cut identical windows instead of each keeping its own arithmetic. The
functions take one chunk row -- a row of chunks.csv, with col_off, row_off,
width_px and height_px -- and an open rasterio dataset on the full grid, and
return rasterio Windows in that raster's own coordinates.
"""

from rasterio.windows import Window

__all__ = ["chunk_window", "chunk_strip_windows"]


def chunk_window(chunk, src):
    """The part of a full-grid raster covered by one chunk, clipped to the raster.

    Plain arithmetic rather than `Window.intersection`, which raises on a window
    that misses the raster entirely. Here a non-overlapping chunk comes back as
    an empty window, and the strip loop simply yields nothing.
    """
    left = max(int(chunk.col_off), 0)
    top = max(int(chunk.row_off), 0)
    right = min(int(chunk.col_off) + int(chunk.width_px), src.width)
    bottom = min(int(chunk.row_off) + int(chunk.height_px), src.height)
    return Window(left, top, max(0, right - left), max(0, bottom - top))


def chunk_strip_windows(chunk, src, n_layers=None, itemsize=4,
                        target_bytes=512 * 2 ** 20):
    """Row strips of one chunk, sized so one strip of all layers stays in memory.

    Yields rasterio Windows in raster coordinates, so each strip can be read with
    `src.read(window=...)` and written straight into the chunk's prediction raster
    at the same offsets. Nothing is resampled: the strip edges are the chunk's own
    pixel edges. `n_layers` defaults to the raster's band count; pass the stack's
    layer count when the stack is assembled from several files.
    """
    if n_layers is None:
        n_layers = src.count
    window = chunk_window(chunk, src)
    if window.width <= 0 or window.height <= 0:
        return
    bytes_per_row = int(window.width) * int(n_layers) * int(itemsize)
    rows = max(1, int(target_bytes // max(1, bytes_per_row)))
    rows = min(rows, int(window.height))
    for offset in range(0, int(window.height), rows):
        yield Window(
            window.col_off, window.row_off + offset,
            window.width, min(rows, int(window.height) - offset),
        )

'''

CHUNK_IO_PATH = os.path.join(OUT_DIR, "chunk_io.py")
with open(CHUNK_IO_PATH, "w") as fh:
    fh.write(CHUNK_IO_SOURCE)
exec(compile(CHUNK_IO_SOURCE, CHUNK_IO_PATH, "exec"))

# ---- Covariate order is part of the model, not an implementation detail. MaxEnt
# names its variables, so a permuted stack fits the wrong layers and still reports
# a clean-looking result. The list is exported in the order the rest of the
# notebook has always used, with the file suffix and the clipping tags stripped so
# the names are readable in a model report.
COVARIATES_PATH = os.path.join(OUT_DIR, "covariates.csv")
COVARIATE_SUFFIXES = (".tiff", ".tif")
COVARIATE_TAGS = ("_clipped_reproj", "_clipped", "_aligned")


def covariate_name(path):
    """A short, stable variable name for one covariate file."""
    name = os.path.basename(path)
    for suffix in COVARIATE_SUFFIXES:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    for tag in COVARIATE_TAGS:
        name = name.replace(tag, "")
    return name


co_covariate_rows = []
for path in COVARIATE_PATHS:
    with rasterio.open(path) as src:
        co_covariate_rows.append({
            "name": covariate_name(path),
            "file": os.path.basename(path),
            "path": path,
            "band": 1,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "res_x": src.res[0],
            "res_y": src.res[1],
        })
covariates = pd.DataFrame(co_covariate_rows)
covariates.to_csv(COVARIATES_PATH, index=False)

# ---- One file that names everything, so the model notebook never hardcodes a path
# it would have to keep in step with this one by hand. The files it lists sit next
# to it, so the notebook can locate them from the manifest's own path; only the
# covariates are absolute, because they live in the Kaggle input dataset.
MANIFEST_PATH = os.path.join(OUT_DIR, "manifest.json")
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_notebook": "data_prep.ipynb",
    "crs": DST_CRS,
    "grid": {
        "bounds": [float(v) for v in te_bounds],
        "res_m": GRID_RES,
        "width_px": int(width),
        "height_px": int(height),
        "transform": [float(v) for v in transform][:6],
    },
    "chunks": {
        "table": "chunks.csv",
        "polygons": "chunks.gpkg",
        "settings": "chunk_settings.csv",
        "cols": int(CHUNK_COLS),
        "rows": int(CHUNK_ROWS),
        "n": int(len(chunks)),
        "id_column": "chunk_id",
        "window_columns": ["col_off", "row_off", "width_px", "height_px"],
    },
    "points": {
        "presences": "presences_chunked.gpkg",
        "background": "background_chunked.gpkg",
        "n_presences": int(len(presences)),
        "n_background": int(len(background)),
        "id_column": "chunk_id",
    },
    "covariates": {
        "table": "covariates.csv",
        "n_layers": int(len(COVARIATE_PATHS)),
        "order": covariates["name"].tolist(),
    },
    "helpers": {
        "module": "chunk_io.py",
        "functions": ["chunk_window", "chunk_strip_windows"],
    },
    "strip_target_bytes": int(STRIP_TARGET_BYTES),
    "thinning_m": float(THIN_DISTANCE_M),
    "background_seed": int(BACKGROUND_SEED),
    "note": (
        "Every file name here is relative to the folder holding this manifest; the "
        "covariate paths are absolute. Read the covariate order from covariates.csv: "
        "fitting a permuted stack fits the wrong layers."
    ),
}
with open(MANIFEST_PATH, "w") as fh:
    json.dump(manifest, fh, indent=2)

print(f"exported       : {OUT_DIR}")
print(f"  manifest.json   grid, chunks, points and paths for the model notebook")
print(f"  covariates.csv  {len(covariates)} layers in fitting order: "
      + ", ".join(covariates["name"]))
print(f"  chunk_io.py     {len(CHUNK_IO_SOURCE.splitlines())} lines: "
      "chunk_window, chunk_strip_windows")
print(f"model notebook : reads manifest.json, then chunks.csv and covariates.csv "
      "from the same folder")


In [ ]:
# ---- Plan step 4c: where the compute goes.
#
# The region is ~2 billion 30 m pixels. Predicting all of it once per fold means
# re-reading every covariate and pushing all those pixels through the model N
# times, which is the dominant cost of the whole project -- far more than fitting.
# So step 5 predicts one chunk at a time, and the rasters are read only for the
# chunk being predicted.
#
# One chunk stack is still large: 15,000 x 29,400 px x 8 layers x 4 bytes is about
# 14 GB, and the model wants more than one copy of that. The read is therefore split
# into row strips a few hundred megabytes each, which is also the shape the
# prediction loop wants: read a strip of all layers, score it, write it into the
# chunk's output raster, drop it.
# The window and strip arithmetic lives in `chunk_io.py`, written by the cell
# above and imported unchanged by the model notebook, so both cut the same
# windows. What is left here is the price of the choice, before any raster is
# opened: a full-chunk stack per fold, and the strip count that keeps each read
# bounded.
stack_gb = (
    chunk_table["width_px"] * chunk_table["height_px"]
    * len(COVARIATE_PATHS) * 4 / 1e9
)
rows_per_strip = np.maximum(
    1, STRIP_TARGET_BYTES // (chunk_table["width_px"] * len(COVARIATE_PATHS) * 4)
)
strips = np.ceil(chunk_table["height_px"] / rows_per_strip).astype(int)
print(f"covariates     : {len(COVARIATE_PATHS)} layers, read per chunk not per region")
print(f"chunk stack    : {stack_gb.min():.1f}-{stack_gb.max():.1f} GB float32 "
      f"(one chunk at a time)")
print(f"strip size     : {strips.min()}-{strips.max()} strips of "
      f"<= {STRIP_TARGET_BYTES / 2**20:.0f} MB per chunk")
print(f"total read     : about one full-region pass for all {len(chunks)} folds, "
      f"instead of one per fold over the whole region")


In [ ]:
# ---- The chunks on the map.
#
# One rectangle per chunk with its presence and background counts, so the audit
# can be checked by eye: a chunk whose box is mostly empty corner (Lake Michigan,
# Lake Erie, the ground north of the states) should be the one that was merged
# away, and no chunk should be a sliver. The chunk boundaries are drawn over the
# state outlines because they ignore state lines entirely -- that is the point of
# the grid, and it is also why a fold can hold parts of two states.
fig, ax = plt.subplots(figsize=(12, 10))

state_outlines.boundary.plot(ax=ax, color="black", linewidth=0.8)
chunk_colors = plt.cm.tab20(np.linspace(0, 1, len(chunk_table)))
for chunk, color in zip(chunk_table.itertuples(), chunk_colors):
    ax.add_patch(plt.Rectangle(
        (chunk.left, chunk.bottom),
        chunk.right - chunk.left, chunk.top - chunk.bottom,
        facecolor=color, alpha=0.12, edgecolor="black", linewidth=1.2,
    ))
    ax.text(
        (chunk.left + chunk.right) / 2, (chunk.bottom + chunk.top) / 2,
        f"chunk {chunk.chunk_id}\n{chunk.n_presences:,} pres\n"
        f"{chunk.n_background:,} bg",
        ha="center", va="center", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  alpha=0.85, edgecolor="0.7"),
    )

ax.scatter(background_x, background_y, s=1.0, color="tab:blue", alpha=0.15,
           linewidths=0)
ax.scatter(presence_x, presence_y, s=5, color="crimson", alpha=0.8, linewidths=0)

ax.set_xlim(te_bounds[0], te_bounds[2])
ax.set_ylim(te_bounds[1], te_bounds[3])
ax.set_aspect("equal")
ax.set_xlabel("Easting (m, EPSG:5070)")
ax.set_ylabel("Northing (m, EPSG:5070)")
ax.set_title(f"{len(chunk_table)} spatial chunks "
             f"({CHUNK_COLS} x {CHUNK_ROWS} grid) for leave-one-chunk-out")

handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson",
           markersize=6, label=f"presences ({len(presences):,})"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue",
           markersize=6, label=f"background ({len(background):,})"),
]
ax.legend(handles=handles, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()
